# NumPy — Coding Challenges

This notebook is a benchmarking ground. Ten challenges, each a small, self-contained problem with the phrasing you would meet in real interviews and competitive screens. Every solution ends with an `assert`, so you can verify your own attempt instantly: if the assertion passes, your code is correct.

The challenges deliberately reuse the whole NumPy curriculum — creating and shaping arrays, broadcasting, boolean masks, reductions, unique counts, sorting, softmax, and outlier clipping. What makes them worth your time is that they are posed backwards: instead of saying 'use indexing', they describe a goal and leave the choice of tool up to you. Recognising which function or pattern each one wants is the skill being measured.

The ten targets:

1. Build a 5x5 identity matrix _without_ `np.eye` — you will need index-array assignment.
2. Normalise every row to unit length with `np.linalg.norm` and `axis=1`, `keepdims=True`.
3. Find the two closest points using a broadcast pairwise-distance matrix.
4. Replace NaNs with their column mean using `np.nanmean`, `np.where`, and `np.take`.
5. Compute a rolling mean via `np.convolve`.
6. Count unique values and their frequencies with `np.unique(return_counts=True)`.
7. Sort a 2D array by its second column using `argsort`.
8. Write a numerically stable softmax that subtracts the max before exponentiating.
9. Generate an 8x8 checkerboard from `np.indices`. 
10. Clip outliers to the 1st and 99th percentiles with `np.clip`.

A habit worth building: read the challenge, then ask yourself which concept from Sessions 13-15 it exercises before writing any code. Each challenge cell contains a solution plus a self-checking assert. Try solving each from memory first — then run the cell to see whether you earned the tick. For any challenge that takes more than two tries, read the assert carefully: it usually tells you exactly which constraint your code missed.

## Setup

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 30)
np.random.seed(0)
print('pandas', pd.__version__)

## Challenge 1

Create a 5x5 identity matrix without np.eye.

## 1.1 Hint — Identity without np.eye

The challenge forbids `np.eye`, so build the matrix the same way you would fill any pattern: start from a zero array `np.zeros((5, 5), dtype=int)` and write 1s onto the diagonal in one shot. The diagonal cells are exactly the positions `(i, i)`, so index with two arrays of the same length — `m[np.arange(5), np.arange(5)] = 1` — and NumPy assigns elementwise, row-by-row from the paired indices. This is the canonical example of index-array assignment, the seed of all fancy-indexing tricks.

The assert compares your matrix to `np.eye(5, dtype=int)`, so ensure your zeros are integer-valued.

In [ ]:
import numpy as np
m = np.zeros((5, 5), dtype=int)
m[np.arange(5), np.arange(5)] = 1
assert np.array_equal(m, np.eye(5, dtype=int))
print(m)

## Challenge 2

Normalise each row of a matrix to unit length.

## 2.1 Hint — Row normalisation

The goal is to make each row's Euclidean length equal to 1. Compute the per-row norm with `np.linalg.norm(X, axis=1, keepdims=True)` — the `keepdims=True` is the crucial detail because it keeps the norms as a `(4, 1)` column so they broadcast across the columns when you divide. Write `unit = X / norms`. The assert checks `np.linalg.norm(unit, axis=1)` is all-close to 1, so use `np.allclose` as a gentle introduction to float comparisons.

In [ ]:
import numpy as np
X = np.random.rand(4, 3)
unit = X / np.linalg.norm(X, axis=1, keepdims=True)
assert np.allclose(np.linalg.norm(unit, axis=1), 1)
print(unit.round(3))

## Challenge 3

Find the two closest points in a small set.

## 3.1 Hint — Closest pair

'The two closest points' screams pairwise distances. Compute them with a broadcast: `p[:, None] - p[None, :]` creates a `(n, n, 2)` grid where entry `[i, j]` holds point-i minus point-j; square, sum over the last axis, and `np.sqrt`. The self-distances on the diagonal are 0, so use `np.fill_diagonal(d, np.inf)` to ignore them. Then `d.argmin()` gives the flat position of the smallest distance and `np.unravel_index` converts it into `(i, j)` coordinates. Practise this broadcast-distance idiom — it underpins k-nearest neighbours.

In [ ]:
import numpy as np
p = np.array([[0., 0.], [1., 1.], [5., 5.]])
d = np.sqrt(((p[:, None] - p[None]) ** 2).sum(-1))
np.fill_diagonal(d, np.inf)
i, j = np.unravel_index(d.argmin(), d.shape)
assert sorted([i, j]) == [0, 1]
print(i, j)

## Challenge 4

Replace all NaN values with the column mean.

## 4.1 Hint — NaN to column mean

The plan has three steps. Compute column means while skipping NaN with `np.nanmean(X, axis=0)`. Locate the missing cells with `np.isnan(X)`; `np.where(np.isnan(X))` returns a tuple of (row indices, column indices). Then write the mean of each column into its NaN slots: index with that tuple and assign the gathered means — `np.take(means, idx[1])` fetches exactly one mean per missing column. The assert confirms no NaN survives and nothing else changed.

In [ ]:
import numpy as np
X = np.array([[1., np.nan], [3., 4.]])
means = np.nanmean(X, axis=0)
idx = np.where(np.isnan(X))
X[idx] = np.take(means, idx[1])
assert not np.isnan(X).any()
print(X)

## Challenge 5

Compute a rolling mean with stride tricks or convolution.

## 5.1 Hint — Rolling mean by convolution

A rolling mean over a window of 3 is a convolution with a kernel of three equal weights that sum to 1: `np.ones(3) / 3`. When you `np.convolve(a, kernel, mode='valid')`, every output position is the dot product of the kernel with the length-3 slice ending there. The 'valid' mode drops the partial windows at the ends, so for seven inputs you get five outputs. Convolution is how moving averages, smoothing, and blur are actually implemented — well worth absorbing here.

In [ ]:
import numpy as np
a = np.arange(1, 8, dtype=float)
roll = np.convolve(a, np.ones(3) / 3, mode='valid')
assert np.isclose(roll[0], 2.0)
print(roll.round(2))

## Challenge 6

Count unique values and their frequencies.

## 6.1 Hint — Unique values and counts

The one-call answer is `np.unique(a, return_counts=True)`, which returns a tuple of the sorted distinct values and, in parallel, how many times each appears. Because both lists are positionally aligned, `dict(zip(vals.tolist(), counts.tolist()))` builds a clean frequency table. This is the NumPy precursor to pandas' `value_counts`, and it handles the duplicate-heavy arrays like `[1, 2, 2, 3, 3, 3]` (counts `[1, 2, 3]`) without writing any counting loop.

In [ ]:
import numpy as np
a = np.array([1, 2, 2, 3, 3, 3])
vals, counts = np.unique(a, return_counts=True)
assert counts.tolist() == [1, 2, 3]
print(dict(zip(vals.tolist(), counts.tolist())))

## Challenge 7

Sort a 2-D array by its second column.

## 7.1 Hint — Sort by second column

Sorting a table by a key column is a classic `argsort` problem: `X[:, 1]` extracts column 1, `np.argsort(X[:, 1])` returns the row order that would sort it, and indexing `X[order]` reorders the whole matrix by that key. This is different from `np.sort` (which sorts values within each row) — you want to permute whole rows. The assert checks the first row of the result has the smallest second-column value.

In [ ]:
import numpy as np
X = np.array([[3, 9], [1, 2], [5, 5]])
out = X[X[:, 1].argsort()]
assert out[0, 1] == 2
print(out)

## Challenge 8

Compute the softmax of a vector in a numerically stable way.

## 8.1 Hint — Numerically stable softmax

The naive `np.exp(z) / np.exp(z).sum()` overflows when `z` contains large values — `exp(1000)` is astronomically large. The stable identity is that softmax is unchanged if you shift every logit by the same constant, so subtract the maximum first: `e = np.exp(z - z.max())`, then `e / e.sum()`. The assert verifies the probabilities still sum to 1. This shift-invariance trick appears verbatim in every serious deep learning library.

In [ ]:
import numpy as np
z = np.array([2.0, 1.0, 0.1])
e = np.exp(z - z.max())
soft = e / e.sum()
assert np.isclose(soft.sum(), 1)
print(soft.round(4))

## Challenge 9

Build a checkerboard pattern of shape 8x8.

## 9.1 Hint — Checkerboard from indices

The sum of a cell's row and column indices is even on light cells and odd on dark cells — that parity is the whole pattern. `np.indices((8, 8))` returns two 8x8 grids containing the row and column numbers; summing them along `axis=0` gives `row + col` at every cell, and taking modulo 2 turns that into a checkerboard. The asserts check `board[0,0] == 0` and `board[0,1] == 1`, which settles whether the light or dark colour lands in the corner.

In [ ]:
import numpy as np
board = np.indices((8, 8)).sum(axis=0) % 2
assert board[0, 0] == 0 and board[0, 1] == 1
print(board)

## Challenge 10

Clip outliers to the 1st and 99th percentile.

## 10.1 Hint — Percentile clipping

Identify the tail bounds first, then clamp. `np.percentile(a, [1, 99])` returns the 1st and 99th percentiles in one call. `np.clip(a, lo, hi)` then pulls every value below `lo` up to `lo` and every value above `hi` down to `hi` — classic outlier winsorisation. Because the data is drawn from `rng.normal`, the extremes come from the distribution's long tails, and clipping stabilises downstream statistics like means and correlations. The assert confirms no value escapes the interval.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
a = rng.normal(size=1000)
lo, hi = np.percentile(a, [1, 99])
c = np.clip(a, lo, hi)
assert c.min() >= lo and c.max() <= hi
print(round(float(lo), 3), round(float(hi), 3))

## Summary

These ten challenges measured the full sweep of your NumPy toolkit. You reconstructed an identity matrix with index-array assignment instead of `np.eye`, normalised rows through `keepdims=True` broadcasting, built a pairwise-distance grid to find the closest two points, and repaired missing values with column-mean `np.take`. A rolling mean was one convolution, unique values one `return_counts` call, and a table sort one `argsort`. The stable softmax re-centred logits by their maximum before exponentiating, an `np.indices` grid became a checkerboard, and percentile clipping contained the tails. After each challenge, time yourself: a realistic screening pace is around ten minutes per problem. Re-do the ones that took longer until they fall near five — that speed is what separates knowing the functions from being fluent with them.